In [1]:
import qutip
import qutip_jax

with qutip.CoreOptions(default_dtype="jax"):
    H = qutip.rand_herm(5)
    c_ops = [qutip.destroy(5)]
    rho0 = qutip.basis(5, 4)

result = qutip.mesolve(H, rho0, [0, 1], c_ops=c_ops, options={"method": "diffrax"})

/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/equinox/_jit.py:55: UserWarning: Complex dtype support in Diffrax is a work in progress and may not yet produce correct results. Consider splitting your computation into real and imaginary parts instead.
  out = fun(*args, **kwargs)


In [2]:
result

<Result
  Solver: mesolve
  Solver stats:
    method: 'diffrax: Tsit5()'
    init time: 0.09609127044677734
    preparation time: 0.4245262145996094
    run time: 0.9744548797607422
    solver: 'Master Equation Evolution'
    num_collapse: 1
  Time interval: [0, 1] (2 steps)
  Number of e_ops: 0
  States saved.
>

In [3]:
import jax.numpy as jnp
import qutip
import qutip_jax

N = 10000
tlist = jnp.linspace(0.0, 10.0, 200)
# ``jaxdia`` operators support higher dimensional Hilbert spaces in the GPU
with qutip.CoreOptions(default_dtype="jaxdia"):
    a = qutip.tensor(qutip.qeye(2), qutip.destroy(N))
    sm = qutip.tensor(qutip.destroy(2), qutip.qeye(N))
H = 2.0 * jnp.pi * a.dag() * a + 2.0 * jnp.pi * sm.dag() * sm + 2.0 * jnp.pi * 0.25 * (sm * a.dag() + sm.dag() * a)
# using ``jax`` dtype since ``DiffraxIntegrator`` anyway converts the final state to ``jax``
state = qutip.tensor(qutip.fock(2, 0, dtype="jax"), qutip.fock(N, 8, dtype="jax"))
c_ops = [jnp.sqrt(0.1) * a]
e_ops = [a.dag() * a, sm.dag() * sm]
result = qutip.mcsolve(H, state, tlist, c_ops, e_ops, ntraj=200, options={
    "method": "diffrax"
})

/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/qutip/solver/solver_base.py:576: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(


10.0%. Run time:  37.07s. Est. time left: 00:00:05:33
20.0%. Run time:  71.96s. Est. time left: 00:00:04:47
30.0%. Run time: 107.44s. Est. time left: 00:00:04:10
40.0%. Run time: 142.54s. Est. time left: 00:00:03:33
50.0%. Run time: 177.52s. Est. time left: 00:00:02:57
60.0%. Run time: 212.56s. Est. time left: 00:00:02:21
70.0%. Run time: 248.16s. Est. time left: 00:00:01:46
80.0%. Run time: 283.78s. Est. time left: 00:00:01:10
90.0%. Run time: 318.79s. Est. time left: 00:00:00:35
100.0%. Run time: 353.73s. Est. time left: 00:00:00:00
Total run time: 355.44s


In [6]:
from qutip_jax.qobjevo import JaxJitCoeff
H_0 = 2.0 * jnp.pi * a.dag() * a + 2.0 * jnp.pi * sm.dag() * sm
H_1_op = sm * a.dag() + sm.dag() * a
H = [H_0, [H_1_op, JaxJitCoeff(lambda t, omega: 2.0 * jnp.pi * 0.25 * jnp.cos(2.0 * omega * t), args={
    "omega": 1.0 # arguments for the coefficient function are passed here
}, static_argnames=("omega", ))]]
result = qutip.mcsolve(H, state, tlist, c_ops, e_ops, ntraj=10, options={
    "method": "diffrax"
})

10.0%. Run time:   0.00s. Est. time left: 00:00:00:00
20.0%. Run time:   3.42s. Est. time left: 00:00:00:13
30.0%. Run time:   5.14s. Est. time left: 00:00:00:11
40.0%. Run time:   6.83s. Est. time left: 00:00:00:10
50.0%. Run time:   8.52s. Est. time left: 00:00:00:08
60.0%. Run time:  10.19s. Est. time left: 00:00:00:06
70.0%. Run time:  11.99s. Est. time left: 00:00:00:05
80.0%. Run time:  13.67s. Est. time left: 00:00:00:03
90.0%. Run time:  15.48s. Est. time left: 00:00:00:01
100.0%. Run time:  17.23s. Est. time left: 00:00:00:00
Total run time:  18.92s
